In [2]:
from collections import deque
import heapq

### BFS and DFS from Previous Lab

In [3]:
def read_cube(filename):
    cube=[]
    with open(filename,'r') as f:
        for line in f:
            cube.append(list(map(int,line.strip().split())))
    return cube

def get_neighbors(pos, cube):
    x,y=pos
    dirs=[(1,0),(-1,0),(0,1),(0,-1)]
    n,m=len(cube), len(cube[0])
    neighbors=[]
    for dx,dy in dirs:
        nx,ny=x+dx,y+dy
        if 0<=nx<n and 0<=ny<m and cube[nx][ny]==0:
            neighbors.append((nx,ny))
    return neighbors

def bfs(cube):
    n,m=len(cube),len(cube[0])
    start,goal=(0,0),(n-1,m-1)
    queue=deque([[start]])
    visited=set([start])
    while queue:
        path=queue.popleft()
        x,y=path[-1]
        if (x,y)==goal:
            return path
        for nei in get_neighbors((x,y),cube):
            if nei not in visited:
                visited.add(nei)
                queue.append(path+[nei])
    return -1

def dfs(cube):
    n,m=len(cube),len(cube[0])
    start,goal=(0,0),(n-1,m-1)
    stack=[[start]]
    visited=set([start])
    while stack:
        path=stack.pop()
        x,y=path[-1]
        if (x,y)==goal:
            return path
        for nei in get_neighbors((x,y),cube):
            if nei not in visited:
                visited.add(nei)
                stack.append(path+[nei])
    return -1

cube=read_cube('cube.txt')
print("BFS path:", bfs(cube))
print("DFS path:", dfs(cube))

BFS path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
DFS path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]


### IDS, UCS, Greedy BFS and A* Implemented Below

In [4]:
def dls(cube, path, goal, limit):
    x,y=path[-1]

    if (x,y)==goal:
        return path

    if limit==0:
        return None

    for nei in get_neighbors((x, y), cube):
        if nei not in path:
            result=dls(cube,path+[nei],goal,limit-1)
            if result is not None:
                return result
    
    return None 

def ids(cube):
    n,m=len(cube),len(cube[0])
    start=(0,0)
    goal=(n-1,m-1)
    max_depth=n*m
    
    for depth in range(max_depth):
        result=dls(cube,[start],goal,depth)
        if result is not None:
            return result
    
    return -1

cube=read_cube('cube.txt')
print("IDS path:", ids(cube))

IDS path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]


In [5]:
def ucs(cube):
    n,m=len(cube),len(cube[0])
    start=(0,0)
    goal=(n-1,m-1)

    pq=[(0,[start])]
    visited=set()
    
    while pq:
        cost,path=heapq.heappop(pq)
        x,y=path[-1]
        
        if (x,y)==goal:
            return path
        if (x,y) in visited:
            continue
        visited.add((x,y))
        
        for nei in get_neighbors((x,y), cube):
            if nei not in visited:
                new_cost=cost+1
                heapq.heappush(pq,(new_cost,path+[nei]))
    
    return -1

cube=read_cube('cube.txt')
print("UCS path:", ucs(cube))

UCS path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]


In [6]:
def read_cube(filename):
    cube=[]
    with open(filename,'r') as f:
        for line in f:
            cube.append(list(map(int,line.strip().split())))
    return cube


def get_neighbors2(pos, cube):
    x,y=pos
    dirs=[(1,0),(-1,0),(0,1),(0,-1)]
    n,m=len(cube),len(cube[0])
    neighbors=[]
    
    for dx,dy in dirs:
        nx,ny=x+dx,y+dy
        if 0<=nx<n and 0<=ny<m:
            if cube[nx][ny]==0:
                neighbors.append(((nx,ny),0))
            elif cube[nx][ny]==2:
                neighbors.append(((nx,ny),1))
    
    return neighbors


def heuristic(pos,goal):
    return abs(pos[0]-goal[0])+abs(pos[1]-goal[1])


In [7]:
def greedy_bfs(cube):
    n,m=len(cube),len(cube[0])
    start=(0,0)
    goal=(n-1,m-1)

    pq=[(heuristic(start,goal),[start])]
    visited=set()
    
    while pq:
        h,path=heapq.heappop(pq) 
        x,y=path[-1]
        
        if (x,y)==goal:
            return path
        if (x,y) in visited:
            continue
        visited.add((x,y))
        
        for nei,step_cost in get_neighbors2((x, y), cube):
            if nei not in visited:
                priority=heuristic(nei, goal)
                heapq.heappush(pq,(priority, path + [nei]))
    
    return -1 

cube2 = read_cube('cube2.txt')
result = greedy_bfs(cube2)
print("Greedy BFS path:", result)

Greedy BFS path: [(0, 0), (0, 1), (1, 1), (1, 2), (1, 3), (2, 3), (2, 4), (3, 4), (4, 4)]


In [8]:
def astar(cube):
    n,m=len(cube),len(cube[0])
    start=(0,0)
    goal=(n-1,m-1)

    g=0 
    h=heuristic(start,goal)
    f=g+h
    
    pq=[(f,g,[start])]
    visited=set()
    
    while pq:
        f,g,path=heapq.heappop(pq)
        x,y=path[-1]
        
        if (x,y)==goal:
            return path
        if (x,y) in visited:
            continue
        visited.add((x,y))
        
        for nei, step_cost in get_neighbors2((x,y),cube):
            if nei not in visited:
                new_g=g+step_cost          
                new_h=heuristic(nei,goal)   
                new_f=new_g+new_h          
                heapq.heappush(pq,(new_f,new_g,path+[nei]))
    
    return -1 


cube2=read_cube('cube2.txt')
result=astar(cube2)
print("A* path:", result)

A* path: [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
